In [9]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModel, AutoTokenizer

In [10]:
def get_device():
    if torch.backends.mps.is_available():
        print("Using Apple SIlicon GPU (MPS)")
        return torch.device("mps")
    else:
        print("MPS not available. Falling back to CPU.")
        return torch.device("cpu")


device = get_device()

Using Apple SIlicon GPU (MPS)


In [3]:
MODEL_NAME = "Alibaba-NLP/gte-Qwen2-1.5B-instruct"
print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

Loading model: Alibaba-NLP/gte-Qwen2-1.5B-instruct


In [ ]:
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map="mps",
    low_cpu_mem_usage=True,
)
model.eval()
print("Model loaded successfully.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully.


## 10-K / 10-Q テキストで埋め込みをテスト

`01_fetch_and_chunk.ipynb` で生成済みの `chunks.parquet` を使用する。
これには AAPL / MSFT / GOOGL の 10-K × 5年 + 10-Q × 15四半期 の
Item 1A (Risk Factors) と Item 7 / Part I Item 2 (MD&A) が
FinBERT トークナイザで 510 トークンに分割済みで格納されている。

**gte-Qwen2-1.5B-instruct の特徴**

- Qwen2 (decoder-only) ベース。**last-token pooling** が必須。
- 出力次元 1536、最大入力 8192 トークン。
- クエリ側のみ `"Instruct: {task}\nQuery: {text}"` 形式の指示プレフィックスを付ける。
  ドキュメント側 (10-K 本文) には付けない。
- bfloat16 + MPS で約 3GB VRAM 使用。


In [11]:
# Cell 6: chunks.parquet をロード (01_fetch_and_chunk.ipynb の成果物)
# _helpers の DATA_DIR / CHUNKS_PARQUET を使い、絶対パスを意識せず読む。
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import _helpers

df_chunks = pd.read_parquet(_helpers.CHUNKS_PARQUET)
print("chunks:", len(df_chunks))
print("tickers:", df_chunks["ticker"].unique().tolist())
print("forms:", df_chunks["form"].unique().tolist())
print("sections:", df_chunks["section_key"].unique().tolist())
df_chunks.head(3)

chunks: 1419
tickers: ['AAPL', 'MSFT', 'GOOGL']
forms: ['10-K', '10-Q']
sections: ['item_1a', 'item_7']


,filing_id,section_key,chunk_idx,text,token_count,ticker,form,filing_date
0,0000320193-25-000079,item_1a,0,item 1a. risk factors the following summarizes...,510,AAPL,10-K,2025-10-31
1,0000320193-25-000079,item_1a,1,"financial market volatility, declines in incom...",510,AAPL,10-K,2025-10-31
2,0000320193-25-000079,item_1a,2,"or exports of goods, technology or data, can m...",510,AAPL,10-K,2025-10-31


In [ ]:
from IPython.display import Markdown

sample_text = df_chunks.loc[0, "text"]
display(Markdown(sample_text))

item 1a. risk factors the following summarizes factors that could have a material adverse effect on the company ’ s business, reputation, results of operations, financial condition and stock price. the company may not be able to accurately predict, control or mitigate these risks. statements in this section are based on the company ’ s beliefs and opinions regarding matters that could materially adversely affect the company in the future and are not representations as to whether such matters have or have not occurred previously. the risks and uncertainties described below are not exhaustive and should not be considered a complete statement of all potential risks or uncertainties that the company faces or may face in the future. this section should be read in conjunction with part ii, item 7, “ management ’ s discussion and analysis of financial condition and results of operations ” and the consolidated financial statements and accompanying notes in part ii, item 8, “ financial statements and supplementary data ” of this form 10 - k. macroeconomic and industry risks the company ’ s operations and performance depend significantly on global and regional economic conditions and adverse economic conditions can materially adversely affect the company ’ s business, results of operations, financial condition and stock price. the company has international operations with sales outside the u. s. representing a majority of the company ’ s total net sales. in addition, the company ’ s global supply chain is large and complex and a majority of the company ’ s supplier facilities, including manufacturing and assembly sites, are located outside the u. s. as a result, the company ’ s operations and performance depend significantly on global and regional economic conditions. adverse macroeconomic conditions, including slow growth or recession, high unemployment, inflation, tighter credit, higher interest rates, and currency fluctuations, can adversely impact consumer confidence and spending and materially adversely affect demand for the company ’ s products and services. in addition, consumer confidence and spending can be materially adversely affected in response to changes in fiscal and monetary policy, financial market volatility, declines in income or asset values, and other economic factors. uncertainty about, or a decline in, global or regional economic conditions can also have a significant impact on the company ’ s suppliers, contract manufacturers, logistics providers, distributors, cellular network carriers and other channel partners, and developers. potential outcomes include financial instability ; inability to obtain credit to finance business operations ; and insolvency. adverse economic conditions can also lead to increased credit and collectibility risk on the company ’ s trade receivables ; the failure of derivative counterparties and other financial institutions ; limitations on the company ’ s ability to issue new debt ; reduced liquidity ; and declines in the

In [ ]:
# Cell 7: encoding ヘルパー
# gte-Qwen2 は decoder-only モデルのため pooling は last-token を使う。
# tokenizer は left-padding ではないので、attention_mask.sum(dim=1)-1 の
# 位置 (= 各サンプルの最終 non-pad トークン) を取り出す。
#
# 既知の互換性問題:
#   transformers 4.45+ で DynamicCache.get_usable_length() が廃止された一方、
#   gte-Qwen2 同梱の modeling_qwen.py はまだ古い API を呼ぶ。
#   embedding 用途では KV キャッシュ不要なので use_cache=False を渡して
#   そのコードパスを完全に回避する。
import torch.nn.functional as F
from tqdm.auto import tqdm


def last_token_pool(
    last_hidden_states: torch.Tensor, attention_mask: torch.Tensor
) -> torch.Tensor:
    """各サンプルで最終 non-pad トークンの hidden state を取り出す."""
    left_padding = bool((attention_mask[:, -1].sum() == attention_mask.shape[0]).item())
    if left_padding:
        return last_hidden_states[:, -1]
    seq_lens = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden_states.shape[0]
    return last_hidden_states[
        torch.arange(batch_size, device=last_hidden_states.device), seq_lens
    ]


def get_detailed_instruct(task_description: str, query: str) -> str:
    """クエリ側のみに付ける指示プレフィックス. ドキュメント側には付けない."""
    return f"Instruct: {task_description}\nQuery: {query}"


@torch.inference_mode()
def encode_texts(
    texts: list[str],
    *,
    max_length: int = 2048,
    batch_size: int = 4,
) -> np.ndarray:
    """テキストのリストを 1536 次元の正規化済みベクトルにエンコード.

    Parameters
    ----------
    texts : list[str]
        対象テキスト (instruction 付き / 無しは呼び出し側で制御)
    max_length : int
        トークン化時の上限。10-K チャンクは 510 トークン程度だが、
        MPS のメモリ節約のため 2048 で打ち切る (モデル上限は 8192)。
    batch_size : int
        MPS では 1.5B モデルを bfloat16 でも 4 程度が安全。
    """
    all_vecs: list[np.ndarray] = []
    for start in tqdm(range(0, len(texts), batch_size), desc="encode"):
        batch = texts[start : start + batch_size]
        enc = tokenizer(
            batch,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(device)
        # use_cache=False で modeling_qwen.py の DynamicCache 互換性問題を回避
        out = model(**enc, use_cache=False, return_dict=True)
        vecs = last_token_pool(out.last_hidden_state, enc["attention_mask"])
        vecs = F.normalize(vecs, p=2, dim=1)
        all_vecs.append(vecs.to(torch.float32).cpu().numpy())
    return np.vstack(all_vecs)


print("helpers ready")

helpers ready


In [13]:
# Cell 8: 動作確認 - サンプル 2 件でエンコード + 次元と類似度確認
# AAPL と MSFT の最新 10-K Item 1A から 1 chunk ずつ取り出して比較。
sample_pair = (
    df_chunks[(df_chunks["form"] == "10-K") & (df_chunks["section_key"] == "item_1a")]
    .sort_values(["ticker", "filing_date"])
    .groupby("ticker")
    .tail(1)
    .loc[lambda d: d["ticker"].isin(["AAPL", "MSFT"])]
    .reset_index(drop=True)
)
print(sample_pair[["ticker", "filing_date", "chunk_idx", "token_count"]])

sample_vecs = encode_texts(sample_pair["text"].tolist(), batch_size=2)
print("shape:", sample_vecs.shape)  # 期待: (2, 1536)
cos_aapl_msft = float(sample_vecs[0] @ sample_vecs[1])
print(f"cosine(AAPL Item 1A latest, MSFT Item 1A latest) = {cos_aapl_msft:.4f}")

  ticker filing_date  chunk_idx  token_count
0   AAPL  2025-10-31         30          291
1   MSFT  2025-07-30         30          333


encode:   0%|          | 0/1 [00:00<?, ?it/s]

shape: (2, 1536)
cosine(AAPL Item 1A latest, MSFT Item 1A latest) = 0.4934


In [14]:
# Cell 9: AAPL / MSFT のみに絞って全 chunk をエンコード (時間とメモリ節約)
# GOOGL も含めたい場合は TARGETS に追加するだけ。
TARGETS = ["AAPL", "MSFT"]
df_target = df_chunks[df_chunks["ticker"].isin(TARGETS)].reset_index(drop=True)
print("target chunks:", len(df_target))

vectors = encode_texts(df_target["text"].tolist(), batch_size=4)
print("vectors shape:", vectors.shape)

df_emb = df_target[
    ["filing_id", "ticker", "form", "section_key", "chunk_idx", "filing_date"]
].copy()
df_emb["vector"] = list(vectors)
# 03_embedding_analysis.ipynb と区別するため別ファイルに保存
out_path = _helpers.DATA_DIR / "embeddings_gte_qwen2.parquet"
df_emb.to_parquet(out_path)
print("saved:", out_path, "rows:", len(df_emb))

target chunks: 682


encode:   0%|          | 0/171 [00:00<?, ?it/s]

vectors shape: (682, 1536)
saved: /Users/yukihata/Desktop/quants/notebook/FILING_NLP/data/embeddings_gte_qwen2.parquet rows: 682


In [15]:
# Cell 10: filing × section 単位で平均プーリング → 銘柄間 / 年次の類似度
# 各 chunk ベクトルを正規化済みなので、平均後に再正規化しておく。
def _avg_pool(group: pd.DataFrame) -> np.ndarray:
    v = np.vstack(group["vector"].values).mean(axis=0)
    n = np.linalg.norm(v)
    return v / n if n > 0 else v


pooled = (
    df_emb.groupby(["ticker", "form", "section_key", "filing_id", "filing_date"])
    .apply(lambda g: pd.Series({"mean_vec": _avg_pool(g)}), include_groups=False)
    .reset_index()
    .sort_values(["ticker", "form", "section_key", "filing_date"])
    .reset_index(drop=True)
)
print("pooled rows:", len(pooled))

# 10-K Item 1A の AAPL vs MSFT 最新比較
latest_risk = (
    pooled[(pooled["form"] == "10-K") & (pooled["section_key"] == "item_1a")]
    .groupby("ticker")
    .tail(1)
    .reset_index(drop=True)
)
aapl_vec = latest_risk.loc[latest_risk["ticker"] == "AAPL", "mean_vec"].iloc[0]
msft_vec = latest_risk.loc[latest_risk["ticker"] == "MSFT", "mean_vec"].iloc[0]
print(
    f"cosine(AAPL latest 10-K Item 1A, MSFT latest 10-K Item 1A) = {float(aapl_vec @ msft_vec):.4f}"
)

# 同一銘柄の年次変化 (cosine distance to previous filing)
rows = []
for (ticker, form, section), g in pooled.groupby(["ticker", "form", "section_key"]):
    g = g.sort_values("filing_date").reset_index(drop=True)
    for i in range(1, len(g)):
        cos = float(g.loc[i - 1, "mean_vec"] @ g.loc[i, "mean_vec"])
        rows.append(
            {
                "ticker": ticker,
                "form": form,
                "section_key": section,
                "filing_date": g.loc[i, "filing_date"],
                "cos_sim_prev": cos,
                "diff_score": 1.0 - cos,
            }
        )
df_change = pd.DataFrame(rows)
df_change.sort_values(["ticker", "form", "section_key", "filing_date"]).head(20)

pooled rows: 50
cosine(AAPL latest 10-K Item 1A, MSFT latest 10-K Item 1A) = 0.8379


,ticker,form,section_key,filing_date,cos_sim_prev,diff_score
0,AAPL,10-K,item_1a,2022-10-28,0.992145,0.007855
1,AAPL,10-K,item_1a,2023-11-03,0.990072,0.009928
2,AAPL,10-K,item_1a,2024-11-01,0.991058,0.008942
3,AAPL,10-K,item_1a,2025-10-31,0.981350,0.018650
4,AAPL,10-K,item_7,2022-10-28,0.943795,0.056205
5,AAPL,10-K,item_7,2023-11-03,0.924109,0.075891
6,AAPL,10-K,item_7,2024-11-01,0.926176,0.073824
7,AAPL,10-K,item_7,2025-10-31,0.930024,0.069976
8,AAPL,10-Q,item_1a,2022-01-28,0.839290,0.160710
9,AAPL,10-Q,item_1a,2022-04-29,0.997174,0.002826


In [16]:
# Cell 11: instruction-aware 検索のデモ
# gte-Qwen2-instruct はクエリ側に "Instruct: {task}\nQuery: {text}" を
# 付けると検索精度が上がる。ドキュメント側 (10-K 本文) には付けない。
TASK = (
    "Given a financial risk question, retrieve relevant passages from 10-K Risk Factors"
)
QUERIES = [
    "supply chain disruption risk from concentration in China",
    "cybersecurity incidents affecting customer data",
    "foreign exchange rate fluctuation impact on revenue",
]
query_inputs = [get_detailed_instruct(TASK, q) for q in QUERIES]
query_vecs = encode_texts(query_inputs, batch_size=4)

# ドキュメント側は Cell 9 で encode 済みのチャンクをそのまま流用
doc_vecs = np.vstack(df_emb["vector"].values)
sims = query_vecs @ doc_vecs.T  # (n_query, n_doc) コサイン類似度

TOP_K = 3
for q_idx, q in enumerate(QUERIES):
    print(f"\n=== Query: {q!r} ===")
    top_idx = np.argsort(-sims[q_idx])[:TOP_K]
    for rank, doc_idx in enumerate(top_idx, 1):
        meta = df_emb.iloc[doc_idx]
        text_preview = df_target.iloc[doc_idx]["text"][:160].replace("\n", " ")
        print(
            f"  #{rank} score={sims[q_idx, doc_idx]:.4f} "
            f"{meta['ticker']} {meta['form']} {meta['section_key']} "
            f"{meta['filing_date'].date()} chunk={meta['chunk_idx']}"
        )
        print(f"     {text_preview!r}")

encode:   0%|          | 0/1 [00:00<?, ?it/s]


=== Query: 'supply chain disruption risk from concentration in China' ===
  #1 score=0.7073 AAPL 10-K item_1a 2022-10-28 chunk=8
     'in single locations. changes or additions to the company ’ s supply chain require considerable time and resources and involve significant risks and uncertaintie'
  #2 score=0.6977 AAPL 10-K item_1a 2024-11-01 chunk=7
     'defects and experiences unanticipated product defect liabilities from time to time. while the company relies on its partners to adhere to its supplier code of c'
  #3 score=0.6921 AAPL 10-K item_1a 2024-11-01 chunk=2
     'vietnam. restrictions on international trade, such as tariffs and other controls on imports or exports of goods, technology or data, can materially adversely af'

=== Query: 'cybersecurity incidents affecting customer data' ===
  #1 score=0.7165 AAPL 10-K item_1a 2022-10-28 chunk=18
     'confidential information or disrupt normal business operations, and could, among other things, impair the company ’ s ability to